<!-- nav -->
[← 1 · Labelling a shelf: a transform, applied and certified](01_Labelling_A_Shelf.ipynb) · [Home](../../README.md) · [3 · Running again: reruns and diffs, queries over the certificates →](03_Running_Again.ipynb)

# 2 · Choosing what to apply: candidates from a frame stream

**The problem.** In case study 1 you found each frame and decided its case yourself. A generator meets many frames,
and for each, several choices; what it offers must be what the model holds, each choice a candidate that you, or a
policy you chose, decide. As the user put it, "The transform candidates must come from a filter fed by a frame stream.
That's how Store works, no?"

`Transforms.Session(transforms, frames)` is that filter (2.1): `transforms`, visitors of the transforms to consider;
`frames`, your frame stream, each frame a `Visitors.OfPattern`. A follow-up call, `session.ranked(policy)`, ranks each
frame's candidates by a policy (2.2). You take a candidate by applying it with your own visitors, verifying, and
certifying (2.3).

The shelf, its prototype `shelf.Unlabelled` and the procedure `label` are case study 1's, the shelf's in `toolkit`.

In [1]:
import { Builders, Proxies, References } from "@mbse/schemas/Framework";
import { Transforms } from "@mbse/transforms";

import { assignments_of, bear, frames, items, model, shown } from "./toolkit.js";

/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const item = sync.i;
  const name = String(item.name);
  (Proxies.store_of(item) as any).shelf.Item(item, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase()));
}

const transforms: any = model(); // the transforms' store
transforms.transaction(Transforms.install);
transforms.transaction((t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled));
const procedures = new Transforms.Procedures();
procedures.register("label", label);
const certs: any = new Builders.Store(); // the certificates' store
certs.transaction(Transforms.install);

/** Your frame stream, within the store's transaction: each frame found, as a `Visitors.OfPattern`, made as it is
 * pulled, which it prints. */
function* stream(store: any, found: Map<string, any>[]): Generator<any> {
  for (const frame of found) {
    const visitor = bear(store, "shelf.Unlabelled", frame);
    console.log(`pulled ${visitor.i.name}`);
    yield visitor;
  }
}

<a id="T02.1"></a>
## 02.1 · A session: candidates from a frame stream

Within your transactions, make the session with the transforms to consider, in the order that breaks ties, and your
frame stream. The filter is each transform's prototype: a frame passes where it is a frame of the prototype, and
becomes candidates, one per value of each finite parameter (a `bool`'s, or a union of options' names, as `case` is
here), any other parameter left open, for you to answer (`answer`). Iterating the session pulls each frame once, as
you go, and gives that frame's candidates.

"In a streaming system termination conditions are relative": the session ends when your stream does, and queries
nothing again. It consumes its stream, so it is iterated once: iterated again, it gives nothing.

In [2]:
const data: any = model("bolt", "nut");
const found = frames(data, "shelf.Unlabelled");
transforms.transaction(() => data.transaction(() => {
  const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
  const session = new Transforms.Session([Label], stream(data, found));
  console.log("nothing pulled yet");
  for (const candidates of session) {
    for (const candidate of candidates) {
      console.log(`  ${candidate.name} on ${candidate.frame.i.name}: ${shown(candidate.assignments)}`);
    }
  }
  console.log(`iterated again: ${[...session].length} candidates`);
}));

nothing pulled yet
pulled bolt
  Label on bolt: case=upper
  Label on bolt: case=lower
pulled nut
  Label on nut: case=upper
  Label on nut: case=lower
iterated again: 0 candidates


<a id="T02.2"></a>
## 02.2 · A policy ranks each frame's candidates

A policy, `new Transforms.Policy(new Transforms.Clause(transform, { assignments, weight }), ...)`, weighs candidates:
a clause names transforms by object reference (`References.of(store, transform)`), one or several, and applies to a
candidate of one it names whose assignments hold its own. A candidate's score is the sum of the weights of the clauses
that apply; `session.ranked(policy)` gives each frame's candidates by score, highest first, ties in the session's
order.

In [3]:
{
  const data: any = model("bolt", "nut");
  const found = frames(data, "shelf.Unlabelled");
  transforms.transaction(() => data.transaction(() => {
    const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
    const named = References.of(transforms, Label);
    const policy = new Transforms.Policy(new Transforms.Clause(named, { assignments: { case: "lower" }, weight: 2.0 }),
      new Transforms.Clause(named, { assignments: {}, weight: 0.5 }));
    for (const candidates of new Transforms.Session([Label], stream(data, found)).ranked(policy)) {
      for (const candidate of candidates) {
        console.log(`  ${candidate.name} on ${candidate.frame.i.name}: ${shown(candidate.assignments)}, score ${candidate.score}`);
      }
    }
  }));
}

pulled bolt
  Label on bolt: case=lower, score 2.5
  Label on bolt: case=upper, score 0.5
pulled nut
  Label on nut: case=lower, score 2.5
  Label on nut: case=upper, score 0.5


<a id="T02.3"></a>
## 02.3 · Taking a candidate: apply it with your own visitors, verify, certify

A candidate is taken by applying it: `Transforms.apply` with its transform, its assignments and your visitors, its
`frame` among them as the argument you choose (here the sync). Verifying is your own call, between `apply` and
`certify`, with the same visitors: "The pattern constraint - application may optionally verify that the pattern is
falsified by the application or more generally an additional metric that computes recursion progress toward
termination". Where it fails, you don't certify, and no certificate is written. Then `Transforms.certify`, giving
`by`, who decided.

Below, you decide bolt's case yourself and leave nut's to the policy, taking its first.

In [4]:
/** Your verification: the prototype no longer holds of the sync's frame, its item labelled. */
function falsified(sync: any): boolean {
  return sync.i.label.get() !== null;
}

{
  const data: any = model("bolt", "nut");
  const found = frames(data, "shelf.Unlabelled");
  transforms.transaction(() => data.transaction(() => {
    const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
    const named = References.of(transforms, Label);
    const policy = new Transforms.Policy(new Transforms.Clause(named, { assignments: { case: "lower" }, weight: 2.0 }),
      new Transforms.Clause(named, { assignments: {}, weight: 0.5 }));
    for (const candidates of new Transforms.Session([Label], stream(data, found)).ranked(policy)) {
      const [taken, by] = String(candidates[0]!.frame.i.name) === "bolt"
        ? [candidates.find((c) => c.assignments.case === "upper")!, "caller"] // you decide bolt's
        : [candidates[0]!, "policy"]; // and leave the rest to the policy
      Transforms.apply(Label, taken.assignments, procedures, { sync: taken.frame });
      if (falsified(taken.frame)) { // where it fails, you don't certify
        Transforms.certify(certs, Label, taken.assignments, "ranked", by, { sync: taken.frame });
      }
    }
  }));
  items(data);
  certs.transaction(() => {
    const history = certs.singleton("Transforms.TheHistories").histories.at("ranked").history;
    const decisions = [...history.decisions];
    for (const [k, entry] of [...history.steps].entries()) {
      console.log(`step ${entry.index.get()}: ${shown(assignments_of(entry.step))}, decided by ${decisions[k].by}`);
    }
  });
}

pulled bolt
pulled nut
bolt BOLT
nut nut
step 0: case=upper, decided by caller
step 1: case=lower, decided by policy


Case study 3 runs the shelf again, after it changed, reusing these decisions where they still apply, and compares
two runs: both are queries over the certificates.

<!-- nav -->
[← 1 · Labelling a shelf: a transform, applied and certified](01_Labelling_A_Shelf.ipynb) · [Home](../../README.md) · [3 · Running again: reruns and diffs, queries over the certificates →](03_Running_Again.ipynb)